## Chaotic scenario 1: Bank run

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from amm_basics import *
import warnings
warnings.filterwarnings("ignore", message="Discarding nonzero nanoseconds")

# 1. SYNTHETIC TIMELINE & PRICE

n  = 300
ts = pd.date_range("2024-01-01", periods=n, freq="5min").tolist()
p0 = 2000.0   # ETH starting price in DAI

segments = [
    (1.000, 1.000),   # stable
    (1.000, 0.920),   # drop ~8%
    (0.920, 0.920),   # stable
    (0.920, 0.850),   # drop ~7%
    (0.850, 0.850),   # stable
    (0.850, 0.780),   # drop ~8%
    (0.780, 0.780),   # stable
    (0.780, 0.700),   # drop ~10%
    (0.700, 0.700),   # stable
    (0.700, 0.620),   # drop ~11%
    (0.620, 0.620),   # flat bottom
]

seg_len = n // len(segments)
prices  = np.zeros(n)
for idx, (p_from, p_to) in enumerate(segments):
    start = idx * seg_len
    end   = start + seg_len if idx < len(segments) - 1 else n
    size  = end - start
    prices[start:end] = np.linspace(p0 * p_from, p0 * p_to, size) \
                        + np.random.normal(0, p0 * 0.003, size)

# 2. SETUP

eth, dai = AtomicToken("ETH"), AtomicToken("DAI")
arb_wallet = Wallet("Arbitrageur")
lp_wallet  = Wallet("LP")

arb_wallet.deposit(eth, 5000.0)
arb_wallet.deposit(dai, 5_000_000.0)
lp_wallet.deposit(eth,  2000.0)
lp_wallet.deposit(dai,  2_000_000.0)

fee     = 0.003
x_start = 100.0
p_init  = prices[0]

amm_pool  = AMM(eth, dai, UniswapV2(), reserve0=x_start, reserve1=x_start * p_init)
sim_state = State([arb_wallet, lp_wallet], [amm_pool])

POOL_DEAD_THRESHOLD = x_start * 0.05
fixed_trade         = x_start * 0.01   # fixed reference trade for slippage

# 3. SIMULATION

history   = []
lp_eth_0  = amm_pool.reserves[eth]
lp_dai_0  = amm_pool.reserves[dai]
k_0       = lp_eth_0 * lp_dai_0
pool_dead = False

for i in range(n):
    m_price = prices[i]
    res     = amm_pool.reserves

    if pool_dead:
        last = history[-1].copy()
        last['pool_dead'] = True
        history.append(last)
        continue

    # ARBITRAGE
    p_amm = res[dai] / res[eth]
    p_bid = p_amm * (1 - fee)
    p_ask = p_amm / (1 - fee)

    if m_price > p_ask:
        target_x = np.sqrt((res[eth] * res[dai]) / (m_price * (1 - fee)))
        dy_in    = (res[eth] * res[dai] / target_x) - res[dai]
        if dy_in > 0.001:
            try:
                sim_state.swap(Transaction("swap", arb_wallet, dai, eth, dy_in))
            except Exception:
                pass
    elif m_price < p_bid:
        target_x = np.sqrt((res[eth] * res[dai]) / (m_price / (1 - fee)))
        dx_in    = target_x - res[eth]
        if dx_in > 0.001:
            try:
                sim_state.swap(Transaction("swap", arb_wallet, eth, dai, dx_in))
            except Exception:
                pass

    res = amm_pool.reserves

    # LP WITHDRAWAL (bank run cascade)
    price_drop   = (p_init - m_price) / p_init
    panic_factor = max(0, price_drop - 0.10)

    if res[eth] > 0.001 and res[dai] > 0.001:
        ideal_pre  = fixed_trade * (res[dai] / res[eth])
        actual_pre = (fixed_trade * (1 - fee) * res[dai]) / (res[eth] + fixed_trade * (1 - fee))
        slip_pre   = max(0.0, (ideal_pre - actual_pre) / ideal_pre)
    else:
        slip_pre = 1.0

    slip_fear  = min(slip_pre * 10, 1.0)
    p_withdraw = min(panic_factor * 2 + slip_fear * 0.3, 0.95)

    if np.random.random() < p_withdraw and res[eth] > POOL_DEAD_THRESHOLD:
        withdraw_pct = np.random.uniform(0.05, min(0.25, panic_factor + 0.05))
        new_eth = res[eth] * (1 - withdraw_pct)
        new_dai = res[dai] * (1 - withdraw_pct)
        if new_eth > 0.001 and new_dai > 0.001:
            amm_pool.reserves[eth] = new_eth
            amm_pool.reserves[dai] = new_dai

    # SLIPPAGE after withdrawal
    res = amm_pool.reserves
    if res[eth] > 0.001 and res[dai] > 0.001:
        dai_out_ideal  = fixed_trade * (res[dai] / res[eth])
        dai_out_actual = (fixed_trade * (1 - fee) * res[dai]) / (res[eth] + fixed_trade * (1 - fee))
        slippage = max(0.0, (dai_out_ideal - dai_out_actual) / dai_out_ideal)
    else:
        slippage = 1.0

    # METRICS
    current_k  = res[eth] * res[dai]
    m_x        = np.sqrt(current_k / m_price) if m_price > 0 else res[eth]
    x_pure     = np.sqrt(k_0 / m_price)
    v_hodl     = lp_eth_0 * m_price + lp_dai_0
    il_pure    = ((x_pure * m_price + k_0 / x_pure) / v_hodl) - 1
    net_return = ((res[eth] * m_price + res[dai]) / v_hodl) - 1

    if res[eth] < POOL_DEAD_THRESHOLD:
        pool_dead = True

    history.append({
        'price':      m_price,
        'x':          res[eth],
        'y':          res[dai],
        'k':          current_k,
        'm_x':        m_x,
        'm_y':        current_k / m_x if m_x > 0 else res[dai],
        'slippage':   slippage * 100,
        'il_pure':    il_pure * 100,
        'net_return': net_return * 100,
        'liq_eth':    res[eth] / x_start,
        'L':          np.sqrt(current_k),
        'pool_dead':  pool_dead,
    })

h_df = pd.DataFrame(history)

# =========================================================
# 4. INTERPOLATION
# =========================================================
INTERP_STEPS = 5
cols_interp  = ['price', 'x', 'y', 'k', 'm_x', 'm_y',
                'slippage', 'il_pure', 'net_return', 'liq_eth', 'L']

smooth_rows = []
for i in range(len(h_df) - 1):
    ra, rb = h_df.iloc[i], h_df.iloc[i + 1]
    for s in range(INTERP_STEPS):
        t   = s / INTERP_STEPS
        row = {c: ra[c] + (rb[c] - ra[c]) * t for c in cols_interp}
        ts_a = ts[i].timestamp()
        ts_b = ts[min(i + 1, len(ts) - 1)].timestamp()
        row['timestamp'] = pd.Timestamp(ts_a + (ts_b - ts_a) * t, unit='s')
        row['price_idx'] = i + t
        row['pool_dead'] = bool(ra['pool_dead'])
        smooth_rows.append(row)

last = h_df.iloc[-1].to_dict()
last['timestamp'] = ts[-1]
last['price_idx'] = len(h_df) - 1
smooth_rows.append(last)
s_df = pd.DataFrame(smooth_rows)

# 5. RANGES (always after s_df)

x_time_min, x_time_max = ts[0], ts[-1]
y_price_min  = prices.min() * 0.95
y_price_max  = prices.max() * 1.05
slip_max     = max(s_df['slippage'].max() * 1.1, 15)
nr_min       = min(s_df['net_return'].min(), s_df['il_pure'].min()) * 1.2
nr_max       = max(s_df['net_return'].max(), 2)
x_range_full = np.linspace(h_df['x'].min() * 0.3, h_df['x'].max() * 1.2, 300)

L_max = h_df['L'].max() * 1.1

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "<b>ETH/DAI Market Price</b>",
        "<b>CPMM Curve — Liquidity Draining</b>",
        "<b>Pool Liquidity L = √(x·y)</b>",
        "<b>Slippage (%) — Pool Usability</b>",
    ),
    horizontal_spacing=0.12,
    vertical_spacing=0.22,
)

row0 = s_df.iloc[0]
k0   = row0['k']

fig.add_trace(go.Scatter(                                           # 0
    x=[ts[0]], y=[prices[0]],
    mode='lines', line=dict(color='#1E3A5F', width=2),
    name="ETH Price (DAI)"
), row=1, col=1)

fig.add_trace(go.Scatter(                                           # 1
    x=x_range_full, y=k0 / x_range_full,
    line=dict(color='rgba(0,0,0,0.2)', width=2),
    fill='tozeroy', fillcolor='rgba(0,0,0,0.05)',
    name="k curve"
), row=1, col=2)

fig.add_trace(go.Scatter(                                           # 2
    x=[row0['x']], y=[row0['y']], mode='markers',
    marker=dict(color='#16A34A', size=12, line=dict(color='white', width=2)),
    name="Pool State"
), row=1, col=2)

fig.add_trace(go.Scatter(                                           # 3
    x=[row0['m_x']], y=[row0['m_y']], mode='markers',
    marker=dict(color='#2563EB', size=8),
    name="Market Price Point"
), row=1, col=2)

fig.add_trace(go.Scatter(                                           # 4
    x=[ts[0]], y=[row0['L']],
    mode='lines', line=dict(color='#7C3AED', width=2),
    fill='tozeroy', fillcolor='rgba(124,58,237,0.12)',
    name="Liquidity L"
), row=2, col=1)

fig.add_trace(go.Scatter(x=[None], y=[None], showlegend=False))     # 5 (index filler)

fig.add_trace(go.Scatter(                                           # 6
    x=[ts[0]], y=[row0['slippage']],
    mode='lines', line=dict(color='#16A34A', width=2),
    fill='tozeroy', fillcolor='rgba(22,163,74,0.15)',
    name="Slippage (%)"
), row=2, col=2)

fig.add_trace(go.Scatter(x=[None], y=[None], showlegend=False))     # 7
fig.add_trace(go.Scatter(x=[None], y=[None], showlegend=False))     # 8

fig.add_hline(y=3,  line=dict(color='#F59E0B', width=1, dash='dash'),
              annotation_text="3% warning",   row=2, col=2)
fig.add_hline(y=10, line=dict(color='#DC2626', width=1, dash='dash'),
              annotation_text="10% critical", row=2, col=2)

# 7. ANIMATION FRAMES

step_skip = 6
frames    = []

for i in range(0, len(s_df), step_skip):
    row     = s_df.iloc[i]
    k_val   = max(row['k'], 1e-6)
    ri      = int(row['price_idx'])
    ts_i    = s_df['timestamp'][:i + 1].tolist()
    liq     = row['liq_eth']
    slip    = row['slippage']
    is_dead = row['pool_dead']

    alpha   = max(0.08, liq * 0.4)
    curve_c = f'rgba(200,50,50,{alpha:.2f})' if liq < 0.5 \
              else f'rgba(0,0,0,{alpha:.2f})'
    dot_c   = '#DC2626' if is_dead \
              else '#EF4444' if liq < 0.3 \
              else '#F59E0B' if liq < 0.7 \
              else '#16A34A'
    slip_c    = '#DC2626' if slip > 10 else '#F59E0B' if slip > 3 else '#16A34A'
    slip_fill = 'rgba(220,38,38,0.15)' if slip > 10 \
                else 'rgba(245,158,11,0.15)' if slip > 3 \
                else 'rgba(22,163,74,0.15)'

    frames.append(go.Frame(
        data=[
            go.Scatter(                                             # 0
                x=ts[:min(ri + 2, len(ts))],
                y=prices[:min(ri + 2, len(prices))].tolist()
            ),
            go.Scatter(                                             # 1
                x=x_range_full,
                y=(k_val / x_range_full).tolist(),
                line=dict(color=curve_c, width=2),
                fill='tozeroy', fillcolor=curve_c,
            ),
            go.Scatter(                                             # 2
                x=[row['x']], y=[row['y']],
                marker=dict(
                    color=dot_c,
                    size=14 if is_dead else 12,
                    symbol='x' if is_dead else 'circle',
                    line=dict(color='white', width=2)
                )
            ),
            go.Scatter(x=[row['m_x']], y=[row['m_y']]),            # 3
            go.Scatter(                                             # 4
                x=ts_i, y=s_df['L'][:i + 1].tolist(),
                line=dict(color='#7C3AED', width=2),
                fill='tozeroy', fillcolor='rgba(124,58,237,0.12)',
            ),
            go.Scatter(x=[None], y=[None]),                         # 5 (filler)
            go.Scatter(                                             # 6
                x=ts_i,
                y=s_df['slippage'][:i + 1].tolist(),
                line=dict(color=slip_c, width=2),
                fill='tozeroy', fillcolor=slip_fill,
            ),
            go.Scatter(x=[None], y=[None],                         # 7
                       name=f"ETH in pool: {row['x']:.2f}"),
            go.Scatter(x=[None], y=[None],                         # 8
                       name=f"DAI in pool: {row['y']:,.0f}"),
        ],
        name=f"f{i}"
    ))

fig.frames = frames

# 8. LAYOUT

fig.update_layout(
    height=700,
    template="plotly_white",
    font=dict(size=12),
    margin=dict(t=70, b=60, l=55, r=40),
    legend=dict(orientation="h", y=-0.12, x=0.5, xanchor="center"),
    xaxis1=dict(range=[str(ts[0]), str(ts[-1])], type="date", fixedrange=True),
    yaxis1=dict(range=[y_price_min, y_price_max], fixedrange=True, title="Price (DAI/ETH)"),
    xaxis2=dict(range=[x_range_full[0], x_range_full[-1]], fixedrange=True, title="ETH Reserves (x)"),
    yaxis2=dict(range=[0, h_df['y'].max() * 1.1], fixedrange=True, title="DAI Reserves (y)"),
    xaxis3=dict(range=[str(ts[0]), str(ts[-1])], type="date", fixedrange=True),
    yaxis3=dict(range=[0, L_max], fixedrange=True, title="L = √(x·y)"),
    xaxis4=dict(range=[str(ts[0]), str(ts[-1])], type="date", fixedrange=True),
    yaxis4=dict(range=[0, slip_max], fixedrange=True, title="Slippage (%)"),
    updatemenus=[dict(
        type="buttons", showactive=False,
        x=0.0, y=1.07, xanchor="left", yanchor="top",
        buttons=[dict(
            label="▶ Start Simulation",
            method="animate",
            args=[None, {
                "frame": {"duration": 30, "redraw": True},
                "fromcurrent": True,
                "transition": {"duration": 0, "easing": "linear"},
                "mode": "immediate"
            }]
        )]
    )]
)

fig.show()

![Bank Run Simulation](bankrun_simulation.gif)

What we can see in the animation is that the LPs withdraw tokens because the price is going down, this triggers an increase in the slippage, so the pool is increasingly unable to facilitate efficient trades, eventually becoming unusable as slippage reaches critical levels and liquidity collapses entirely.

### Theorical background

In a Constant Product Market Maker (CPMM), the invariant is:

$$
xy = k
$$

After a swap:

$$
(x+\Delta x)(y - \Delta y) = k
$$

Solving for $\Delta y$:

$$
\Delta y = \frac{y \Delta x}{x + \Delta x}
$$

The expected and real prices are:

$$
P_{expected} = \frac{y}{x}, \quad
P_{real} = \frac{\Delta y}{\Delta x}
$$

Substituting:

$$
P_{real} = \frac{y}{x + \Delta x}
$$

Therefore, the slippage is:

$$
\text{Slippage}
=
\frac{P_{expected} - P_{real}}{P_{expected}}
=
\frac{\Delta x}{x + \Delta x}
$$

Hence, for a fixed trade size $\Delta x$, larger reserves imply lower slippage.

#### Liquidity Withdrawal and Pool Collapse

When the market price of the pooled asset declines persistently, rational LPs face
growing impermanent loss. As a result, they begin withdrawing liquidity to limit their
exposure, reducing both $x$ and $y$ proportionally and shrinking $k$:

$$
k' = x'y' < k
$$

From the slippage formula, a reduction in reserves $x$ directly increases slippage for
any fixed trade size $\Delta x$:

$$
\text{Slippage} = \frac{\Delta x}{x + \Delta x} \uparrow \quad \text{as } x \downarrow
$$

This creates a self-reinforcing feedback loop. Higher slippage makes the pool less
attractive to traders, reducing fee revenue. Lower fee revenue further discourages LPs
from maintaining their positions, triggering additional withdrawals. As the cycle
repeats, the pool enters a **liquidity death spiral**:

$$
\text{price} \downarrow
\;\Rightarrow\;
\text{LP withdrawals} \uparrow
\;\Rightarrow\;
x, y \downarrow
\;\Rightarrow\;
\text{slippage} \uparrow
\;\Rightarrow\;
\text{trader demand} \downarrow
\;\Rightarrow\;
\text{fee revenue} \downarrow
\;\Rightarrow\;
\text{LP withdrawals} \uparrow \;\cdots
$$

Note: in the simulation code, LP withdrawals are triggered directly by the price drop,
rather than by the reduction in fee revenue. The fee revenue channel is modelled implicitly
through the price dynamic, keeping the feedback loop analytically clean.

In the limit, as $x \to 0$, slippage approaches $100\%$ and the pool becomes
effectively unusable, unable to facilitate any meaningful trade.


### Extra: Hybrid Function Market Maker (HFMM).

We can use a **hybrid approach between a Constant Sum Market Maker (CSMM) and a Constant Product Market Maker (CPMM)** in order to **reduce slippage for small trades while preventing the pool from running out of reserves**, as can happen in a pure CSMM. This market maker is also known as StableSwap.

This is achieved by blending the invariants of the two AMMs: the CSMM component keeps the price stable for small trades, minimizing slippage, while the CPMM component ensures that the pool remains liquid and that no token is ever completely drained, even under large trades.  

Mathematically, the hybrid invariant can be written as:

$$
k = \lambda (x + y_{\text{normalized}}) + (1 - \lambda) \sqrt{x \cdot y_{\text{normalized}}}
$$

where:

- $x, y$ are the pool reserves of the two tokens,  
- $y_{\text{normalized}} = y / P$ adjusts for price scaling,  
- $\lambda \in [0,1]$ controls the weight between the CSMM and CPMM components.  

For $\lambda \to 1$, the pool behaves more like a CSMM (low slippage for small trades).  
For $\lambda \to 0$, the pool behaves like a CPMM (resilient to large trades, never emptying the reserves).  

This **hybrid AMM** combines the best of both worlds: small slippage for minor transactions and safety against total depletion of any token.


### HFMM Pricing Curve

In [ ]:
import numpy as np
import plotly.graph_objects as go

x0, y0 = 100, 100
lmbda = 0.8

k_target = 100 

x_range = np.linspace(0.1, 500, 300)
y_range = np.linspace(0.1, 500, 300)
X, Y = np.meshgrid(x_range, y_range)

# HFMM Invariant
Z_hfmm = lmbda * (X + Y) / 2 + (1 - lmbda) * np.sqrt(X * Y)

# Constant Sum
y_sum_tangent = 200 - x_range

# Constant Product (Uniswap): x * y = 100^2
y_prod = (100**2) / x_range

# =========================================================
# 3. VISUALIZACIÓN
# =========================================================
fig = go.Figure()

# 1. Constant Sum
fig.add_trace(go.Scatter(
    x=x_range, y=y_sum_tangent, 
    mode='lines', name="CSMM",
    line=dict(color='gray', width=1, dash='dash')
))

# 2. Constant Product
fig.add_trace(go.Scatter(
    x=x_range, y=y_prod, 
    mode='lines', name="CPMM",
    line=dict(color='blue', width=1, dash='dot')
))

# 3. HFMM Hybrid
fig.add_trace(go.Contour(
    x=x_range, y=y_range, z=Z_hfmm,
    contours_coloring='none',
    contours=dict(start=k_target, end=k_target, size=1),
    line=dict(color='purple', width=4),
    name="HFMM"
))

fig.add_trace(go.Scatter(
    x=[x0], y=[y0], mode='markers', name="Equilibrium (100,100)",
    marker=dict(color='black', size=10, symbol='x')
))



fig.update_layout(
    title="<b>HFMM Curve",
    xaxis_title="Reserves Token X",
    yaxis_title="Reserves Token Y",
    template="plotly_white",
    xaxis=dict(range=[0, 400], gridcolor='lightgrey', zeroline=True),
    yaxis=dict(range=[0, 400], gridcolor='lightgrey', zeroline=True),
    showlegend=True,
    width=800, height=700
)

fig.show()

![Hybrid Function Market Maker](hfmm.png)

Here we can see that the HFMM curve lies between the CSMM and CPMM curves. The HFMM curve changes its slope more gradually than the CPMM curve, resulting in lower slippage.

### Conclusion

The bank run example explains that automated market makers (AMMs) are unable to avert a successive breakdown when a sustained downtrend occurs. The link between impermanent loss, liquidity exits and slippage creates a self-reinforcing feedback loop; once established, it is almost impossible to stop. Liquidity providers have no mechanism for stopping exits or widening spreads in a defensive manner, they are entirely passive; the liquidity spiral process will run to completion once there is enough pressure from the price shock on the LPs.

The dual exposure of LPs makes them very fragile. They suffer both directional risk from the depreciating asset and opportunity cost from the impermanent loss as compared to simply HODLing. When both of these risks are exacerbated during a downtrend, the LPs' rational response is to withdraw funds, which ultimately decimates the pool's ability to function when market pressures are at their highest and liquidity is needed most.